In [ ]:
!pip install -q -U langchain langchain-community faiss-cpu pypdf langchain-google-genai

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
os.environ['GOOGLE_API_KEY']=userdata.get("GEMINI_API_KEYS")
EMB_MODEL="gemini-embedding-001"
MODEL="gemini-3.5-flash-lite"

In [ ]:
#1.Prepare the document
#1.1 Upload the document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF NAMED:{pdf_name} is uploaded sucessfully")

Saving government_grants_testing.pdf to government_grants_testing.pdf
PDF NAMED:government_grants_testing.pdf is uploaded sucessfully


In [ ]:
#1.2 Extract text from document
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(pdf_name).load()
print("NUMBER OF PAGES: ",len(docs))
text=""
for d in docs:
  text+=d.page_content+"\n------------\n"
print("NUMBER OF CHARACTERS: ",len(text))
print("FIRST FEW CHARACTERS OF PAGE 1: ",docs[0].page_content)

NUMBER OF PAGES:  1
NUMBER OF CHARACTERS:  2149
FIRST FEW CHARACTERS OF PAGE 1:  Government Grants & Startup Funding — Testing Knowledge Base
Sample dataset for testing a RAG pipeline. Information is based on official Government of India sources. Verify current eligibility and application status on the official scheme portal before using the data for real applications.
 S.No
 Grant / Scheme
 Agency
 Who Can Apply
 Purpose
 Funding / Benefit
 Key Eligibility
 Official Source
1
Startup India Seed Fund
Scheme (SISFS)
DPIIT, Ministry of
Commerce & Industry
DPIIT-recognised startups
Proof of concept, prototype, product trials,
market entry and commercialisation
Up to ■20 lakh grant for
PoC/prototype/product trials; up to ■50
lakh investment for market
entry/commercialisation/scaling
Startup incorporated not more than 2
years at application; innovative, scalable
business idea
seedfund.startupindia.gov.in
2
NIDHI-PRAYAS
Department of Science &
Technology (DST)
Innovators / aspiring
entreprene

In [ ]:
#1.3 Overalaping Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)
chunks=splitter.split_documents(docs)
print("CHUNKING DONE WITH NUMBER OF CHUNKS: ",len(chunks))

CHUNKING DONE WITH NUMBER OF CHUNKS:  4


In [ ]:
#1.3Embeddings Vectors FAISS
from langchain_community.vectorstores import FAISS
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
vectorstore=FAISS.from_documents(chunks,embeddings)


In [ ]:
#1.3 Retreival
retriever=vectorstore.as_retriever(search_kwargs={'k':3})
found=retriever.invoke("SUMMARIZE ME WITH PDF")
for d in found:
    print(d.page_content)
    print("*"*60)

Government Grants & Startup Funding — Testing Knowledge Base
Sample dataset for testing a RAG pipeline. Information is based on official Government of India sources. Verify current eligibility and application status on the official scheme portal before using the data for real applications.
 S.No
 Grant / Scheme
 Agency
 Who Can Apply
 Purpose
 Funding / Benefit
 Key Eligibility
 Official Source
1
Startup India Seed Fund
Scheme (SISFS)
DPIIT, Ministry of
Commerce & Industry
DPIIT-recognised startups
Proof of concept, prototype, product trials,
market entry and commercialisation
Up to ■20 lakh grant for
PoC/prototype/product trials; up to ■50
lakh investment for market
entry/commercialisation/scaling
Startup incorporated not more than 2
years at application; innovative, scalable
************************************************************
technology-based ventures
Entrepreneur-in-Residence support
through participating incubators
Technology-based innovation and
incubation ecosystem requi

In [ ]:
#2.Augumentation and Formating answer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
llm=ChatGoogleGenerativeAI(model=MODEL,max_output_tokens=1500)
prompt=ChatPromptTemplate([
    ('system', """
You are a PDF assistant.

Answer the user's question using ONLY the information provided
in the document context.

Do not invent or add information that is not present in the context.

If the context does not contain enough information, say:
"I could not find enough information in the document."

return clean formatted text so schemes with its details will not get mixed
"""),
    ('human',"context from document: {context} and question:{question}")
])



def format_docs(docs):
  return "\n\n............\n\n".join(d.page_content for d in docs)

rag_chain=(
    {'context':retriever | format_docs,'question':RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [57]:

print("NOTE:Enter exit,bye,quit")
while True:
  user_input=input("Enter your query regarding Government Schemes")
  if user_input.lower() in ['exit','bye','quit']:
    print("EXITED.........THANK YOU")
    break
  response=rag_chain.invoke(user_input)
  print(response)

NOTE:Enter exit,bye,quit
Enter your query regarding Government Schemestell me names of schemes
Based on the provided document, here are the names of the schemes:

1. Startup India Seed Fund Scheme (SISFS)
2. NIDHI-PRAYAS
3. NIDHI-EIR
4. Biotechnology Ignition Grant (BIG)
5. NIDHI Seed Support System (NIDHI-SSS)
Enter your query regarding Government Schemessummarize pdf in 5lines
Here is the summary based on the provided document:

1. The document is a knowledge base of Government Grants and Startup Funding schemes based on official Government of India sources.
2. It details various schemes, including the Startup India Seed Fund Scheme (SISFS) managed by DPIIT, which offers grants up to ■20 lakh and investments up to ■50 lakh for eligible young startups.
3. It outlines the Biotechnology Ignition Grant (BIG) by BIRAC, which provides early-stage grant support for biotechnology innovators and startups.
4. It includes the NIDHI Seed Support System (NIDHI-SSS) by the Department of Science & 